# 00 — What Are Unit Tests? (And Why Should I Care?)

If you've never written a unit test before, this notebook is for you. We'll cover:
- What a unit test actually is
- Why they matter for data engineering
- How pytest works
- How Spark fits into the picture

No code to run here — just concepts. The real action starts in notebook `01`.

## What Is a Unit Test?

A **unit test** is a small piece of code that checks whether another piece of code works correctly.

That's it. You write a function, then you write a test that calls it with known inputs and checks the output.

```python
# Your function
def batting_average(hits, at_bats):
    return round(hits / at_bats, 3)

# Your test
def test_batting_average():
    assert batting_average(150, 500) == 0.300
```

The `assert` keyword says: "this must be True — if it's not, something is broken."

If `batting_average(150, 500)` returns `0.300`, the test **passes**. If it returns anything else, the test **fails**.

## Why Bother? Three Reasons

### 1. Catch bugs before they hit production
If you change your `batting_average` function and accidentally break the math, the test catches it immediately — not three days later when someone notices wrong numbers in a dashboard.

### 2. Documentation that runs
Tests show how your code is *supposed* to work. `test_zero_at_bats_returns_zero()` tells the next person: "hey, we handle the edge case where a player has zero at-bats."

### 3. Confidence to refactor
Want to rewrite a function to be faster? Run the tests. If they all pass, you didn't break anything. If they don't, you know exactly what broke.

## How pytest Works

[pytest](https://docs.pytest.org/) is Python's most popular testing framework. Databricks recommends it for Python unit tests.

The rules are simple:

| Convention | What it means |
|---|---|
| File named `test_*.py` | pytest will discover and run it |
| Function named `test_*` | pytest treats it as a test case |
| `assert <condition>` | If condition is False, the test fails |
| `class Test*` | Group related tests together (optional) |

That's the whole API. No decorators, no inheritance, no boilerplate. Just functions with `assert`.

### Running tests

From your terminal:
```bash
pytest tests/ -v
```

The `-v` flag means "verbose" — it prints the name and result of every test.

From a Databricks notebook:
```python
import pytest, sys
retcode = pytest.main(["tests/", "-v", "-p", "no:cacheprovider"])
assert retcode == 0, "Tests failed!"
```

## Two Kinds of Tests in This Demo

### 1. Pure Python tests (no Spark)

Some functions are just math — they take numbers in and return numbers out. These don't need Spark at all.

```python
def test_batting_average():
    assert batting_average(150, 500) == 0.300
```

These tests run **instantly** — milliseconds. They're the fastest feedback you can get.

### 2. PySpark transformation tests

Other functions take a Spark DataFrame as input and return a transformed DataFrame. To test these, we need a running SparkSession.

```python
def test_adds_batting_avg_column(sample_batting_df):
    result = add_batting_average(sample_batting_df)
    assert "batting_avg" in result.columns
```

We use **Databricks Connect** to get a remote SparkSession from your laptop. Your test code runs locally, but Spark operations execute on a Databricks cluster or serverless compute. The same PySpark code works identically — that's the beauty of Spark's architecture.

## What Are Fixtures?

A **fixture** is pytest's way of providing setup for your tests. Instead of repeating the same setup code in every test, you define it once and pytest injects it automatically.

```python
# In conftest.py (shared across all tests)
@pytest.fixture(scope="session")
def spark():
    session = DatabricksSession.builder.serverless(True).getOrCreate()
    yield session
    session.stop()

@pytest.fixture
def sample_batting_df(spark):
    data = [("Mike Trout", "Angels", 180, 550, ...)]
    return spark.createDataFrame(data, schema)
```

Key ideas:
- `scope="session"` — create the SparkSession once, reuse it for all tests (fast!)
- `yield` — runs the test, then cleans up (stops Spark) after all tests finish
- Fixtures can depend on other fixtures — `sample_batting_df` uses `spark`
- If a test function has a parameter named `spark`, pytest automatically provides it

The `conftest.py` file is special — pytest auto-discovers it. Any fixture defined there is available to all test files without importing.

## Where Does Databricks Fit In?

Databricks supports multiple testing approaches:

| Approach | Where it runs | Spark source | Speed | Best for |
|---|---|---|---|---|
| **Databricks Connect** | Your laptop | Remote cluster/Serverless | Fast (seconds) | Pure logic + transforms |
| **pytest in a notebook** | Databricks cluster | Cluster's SparkSession | Medium | Integration tests |
| **CI/CD pipeline** | GitHub Actions etc. | Databricks Connect | Varies | Automated on every PR |

This demo uses **Databricks Connect** for local testing and shows how to run the same tests in a notebook. The key insight: **write your transformation logic in `.py` files, not directly in notebooks**. Then you can test it from anywhere.

## Project Structure

```
unit-testing-pyspark/
├── baseball_stats.py          ← The code we're testing
├── tests/
│   ├── conftest.py            ← Shared fixtures (SparkSession, sample data)
│   ├── test_pure_python.py    ← Tests that don't need Spark
│   └── test_spark_transforms.py  ← Tests that use DataFrames
├── 00_what_are_unit_tests.ipynb   ← You are here
├── 01_run_tests_locally.ipynb     ← Run tests from your terminal
├── 02_run_tests_in_notebook.ipynb ← Run tests inside Databricks
├── 03_explore_the_data.ipynb      ← Use the functions on real data
└── requirements.txt
```

Next up → [01_run_tests_locally.ipynb](01_run_tests_locally.ipynb)